In [1]:
from datasets import load_dataset

fleurs = load_dataset(
    "google/fleurs",
    "hi_in",
    split="test",
    
)

print(len(fleurs))
print(fleurs[0])

c:\up-JOsh\.venv-1\Lib\site-packages\datasets\load.py:1461: FutureWarning: The repository for google/fleurs contains custom code which must be executed to correctly load the dataset. You can inspect the repository content at https://hf.co/datasets/google/fleurs
You can avoid this message in future by passing the argument `trust_remote_code=True`.
Passing `trust_remote_code=True` will be mandatory to load this dataset from the next major release of `datasets`.
  warnings.warn(


418
{'id': 1766, 'num_samples': 145920, 'path': 'C:\\Users\\Sachin Maurya\\.cache\\huggingface\\datasets\\downloads\\extracted\\aa7974d1b7c6dacaea0a7c52ea562b909ee0eef2ac33d6be4f73ed65a553f182\\10011266027513218401.wav', 'audio': {'path': 'test/10011266027513218401.wav', 'array': array([ 0.        ,  0.        ,  0.        , ..., -0.00048101,
       -0.00045913, -0.00040722], shape=(145920,)), 'sampling_rate': 16000}, 'transcription': 'कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है', 'raw_transcription': 'कुछ अणुओं में अस्थिर केंद्रक होता है, जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है,', 'gender': 0, 'lang_id': 32, 'language': 'Hindi', 'lang_group_id': 4}


In [2]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate
import torch

In [3]:
wer_metric = evaluate.load("wer")

In [4]:
processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").cuda()

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [9]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
import evaluate
import torch
from tqdm import tqdm
import re

wer_metric = evaluate.load("wer")

processor = WhisperProcessor.from_pretrained("openai/whisper-small")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-small").cuda()

forced_decoder_ids = processor.get_decoder_prompt_ids(language="hi", task="transcribe")
model.config.forced_decoder_ids = forced_decoder_ids
model.config.suppress_tokens = []

def normalize(text):
    text = text.lower()
    text = text.replace("।", "")
    text = re.sub(r"[,.!?]", "", text)
    text = re.sub(r"[^\u0900-\u097F\s]", "", text)
    text = re.sub(r"\s+", " ", text)
    text = text.strip()
    return text

predictions = []
references = []

for sample in tqdm(fleurs):
    speech = sample["audio"]["array"]

    inputs = processor.feature_extractor(
        speech,
        sampling_rate=16000,
        return_tensors="pt"
    ).input_features.cuda()

    predicted_ids = model.generate(
        inputs,
        max_length=225
    )

    pred = processor.tokenizer.batch_decode(
        predicted_ids,
        skip_special_tokens=True
    )[0]

    predictions.append(normalize(pred))
    references.append(normalize(sample["transcription"]))

wer_baseline = wer_metric.compute(
    predictions=predictions,
    references=references
)

print("WER:", wer_baseline)

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
100%|██████████| 418/418 [13:45<00:00,  1.97s/it]

WER: 0.6936581516914884


In [12]:
for i in range(4):
    print("Pred:", predictions[i])
    print("Ref :", references[i])
    print()

Pred: अग्वो में आज्द्टर केंद्रख होता है जिसका मतला भी आजा की उन्मे थोडे या बिना किसी जटके से तुटनें की प्रवत्ती होती है
Ref : कुछ अणुओं में अस्थिर केंद्रक होता है जिसका मतलब यह है कि उनमें थोड़े या बिना किसी झटके से टूटने की प्रवृत्ति होती है

Pred: गरीलेंड को बहुत कम जग़़ बसाया गया ता नोर्ष शगास में भे कहते हैं कि एरेक रेड रेद हत्या कि लिए आइस लेंड से निरवासित की आप या आप और आगे पश्च्ट्म की याट्र करते समय गरीलेंड मिला जिसे गरीलेंड नाम दिया गया
Ref : ग्रीनलैंड को बहुत कम जगह बसाया गया था नॉर्स सगास में वे कहते हैं कि एरिक रेड हत्या के लिए आइसलैंड से निर्वासित किया गया था और आगे पश्चिम की यात्रा करते समय ग्रीनलैंड मिला जिसे ग्रीनलैंड नाम दिया गया

Pred: 
Ref : ऐसी कोई वैश्विक परिभाषा नहीं है जिसके लिए निर्मित सामान एंटीक होते हैं कुछ कर एजेंसियां साल से पुराने सामान को एंटीक के तौर पर परिभाषित करती हैं

Pred: तेलीविशन रिपोटो में प्लाँट से निकलने वाला सबे दूए दूए गया हैर
Ref : टेलीविजन रिपोर्टों में प्लांट से निकलने वाला सफेद धुआं दिखाया गया है



In [ ]:
# Number Normalization Technique
units = {
    "शून्य": 0, "एक": 1, "दो": 2, "तीन": 3, "चार": 4,
    "पांच": 5, "छह": 6, "सात": 7, "आठ": 8, "नौ": 9,
    "दस": 10, "ग्यारह": 11, "बारह": 12, "तेरह": 13,
    "चौदह": 14, "पंद्रह": 15, "सोलह": 16, "सत्रह": 17,
    "अठारह": 18, "उन्नीस": 19
}

tens = {
    "बीस": 20, "तीस": 30, "चालीस": 40, "पचास": 50,
    "साठ": 60, "सत्तर": 70, "अस्सी": 80, "नब्बे": 90,
    "पच्चीस": 25, "पैंतीस": 35, "पैंतालीस": 45,
    "पचपन": 55, "पैंसठ": 65, "पचहत्तर": 75,
    "पचासी": 85, "पचानवे": 95
}

multipliers = {
    "सौ": 100,
    "हजार": 1000
}

number_words = list(units.keys()) + list(tens.keys()) + list(multipliers.keys())